In [1]:
import os
import json
from mermaid import Mermaid

In [2]:
from cimgraph.databases.fileparsers import XMLFile
from cimgraph.databases import ConnectionParameters
from cimgraph.models import FeederModel, DistributedArea
# import cimgraph.data_profile.cimhub_2023 as cim 
from cimtopology.distribution import create_distributed_feeder
# import cimgraph.data_profile.cimhub_2023 as cim
import cimgraph.utils as utils
from uuid import UUID

In [3]:
import os
os.environ['CIMG_VALIDATION_LOG_LEVEL'] = 'WARNING'

os.environ['CIMG_CIM_PROFILE'] = 'cimgraph.data_profile.cim18gmdm.canonical'
import cimgraph.data_profile.cim18gmdm.canonical as cim # must match env var

from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
from cimgraph import utils

namespaces={'cim': 'http://cim.ucaiug.io/CIM101/draft#',
            'gmdm': 'http://epri.com/gmdm/2025#',
            'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#'}

file = XMLFile(filename='/home/ande188/GMDM-IOP/test_model/ieee_9500_gmdm.xml', namespaces=namespaces)
network = FeederModel(container=None, connection=file)


In [4]:
for cim_class in network.graph:
    if issubclass(cim_class, cim.Switch):
        for switch in network.list_by_class(cim_class):
            if 'hvmv' in switch.name.lower():
                switch.normalOpen = True

for cim_class in network.graph:
    if issubclass(cim_class, cim.Equipment):
        for equipment in network.list_by_class(cim_class):
            equipment.EquipmentContainer = None
            
for node in network.list_by_class(cim.ConnectivityNode):
    node.ConnectivityNodeContainer = None

In [5]:
for terminal in network.list_by_class(cim.Terminal):
    terminal.__traversed__ = False
    terminal.__is_boundary__ = False

In [6]:
def assign_equipment_container(terminal:cim.Terminal, container:cim.EquipmentContainer):
    node = terminal.ConnectivityNode
    for terminal in node.Terminals:
        if not terminal.__traversed__:
            terminal.__traversed__ = True
            equipment = terminal.ConductingEquipment
            # Assign new equipment container
            if equipment.EquipmentContainer is None:
                equipment.EquipmentContainer = container
            else:
                if isinstance(equipment, cim.Switch):
                    if equipment.EquipmentContainer.uri() != container.uri():
                        equipment.AdditionalEquipmentContainer = container
            # Assign new node container
            if terminal.ConnectivityNode.ConnectivityNodeContainer is None:
                terminal.ConnectivityNode.ConnectivityNodeContainer = container
            # Check if boundary reached
            if isinstance(equipment, cim.Switch):
                if equipment.normalOpen:
                    for next_terminal in equipment.Terminals:
                        next_terminal.__is_boundary__ = True
                    continue
            
            for next_terminal in equipment.Terminals:
                if not next_terminal.__traversed__:
                    if not next_terminal.__is_boundary__:
                        assign_equipment_container(next_terminal, container)

In [7]:
starting_node = network.find_by_attribute(cim.ConnectivityNode, 'name', 'e192860')
feeder = cim.Feeder(name='dso_9500_industrial_district')
starting_terminal = starting_node[0].Terminals[0]
assign_equipment_container(starting_terminal, feeder)


In [8]:
starting_node = network.find_by_attribute(cim.ConnectivityNode, 'name', 'e1047300')
feeder = cim.Feeder(name='dso_9500_node_new_neighborhood')
starting_terminal = starting_node[0].Terminals[0]
assign_equipment_container(starting_terminal, feeder)

In [9]:
starting_node = network.find_by_attribute(cim.ConnectivityNode, 'name', 'e203026')
feeder = cim.Feeder(name='dso_9500_node_old_towne')
starting_terminal = starting_node[0].Terminals[0]
assign_equipment_container(starting_terminal, feeder)

In [10]:
node = network.find_by_attribute(cim.ConnectivityNode, 'name', 'm1069509')
Mermaid(utils.get_mermaid(node))

In [11]:
switch = network.find_by_attribute(cim.Sectionaliser, 'name', 'tsw320328_sw')
Mermaid(utils.get_mermaid(switch[0]))

In [12]:
qwertyuiop

NameError: name 'qwertyuiop' is not defined

In [ ]:
params = ConnectionParameters(filename="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13/IEEE13.xml", cim_profile='cimhub_2023', iec61970_301=8)
xml_file = XMLFile(params)
feeder_mrid = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
feeder = cim.Feeder(mRID = feeder_mrid, name = 'ieee13')
feeder_model = FeederModel(connection=xml_file, container=feeder, distributed=False)

In [ ]:
paths = ["/home/ande188/Powergrid-Models/models/feeders/CIM/XML/ieee123apps/ieee123apps.xml"
    # "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE8500/IEEE8500.xml"
    # "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13/IEEE13.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13_Assets/IEEE13_Assets.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13_OCHRE/IEEE13_OCHRE.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE123/IEEE123.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE123_PV/IEEE123_PV.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE9500bal/IEEE9500bal.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/R2_12_47_2/R2_12_47_2.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/Transactive/Transactive.xml",
    #      "/home/ande188/Powergrid-Models/models/feeders/CIM/XML/ACEP_PSIL/ACEP_PSIL.xml"
         ]

In [ ]:
for filename in paths:
    params = ConnectionParameters(filename=filename, cim_profile='cimhub_2023', iec61970_301=8)
    xml_file = XMLFile(params)
    feeder = cim.Feeder()
    feeder_model = FeederModel(connection=xml_file, container=feeder, distributed=False)
    feeder_model.container = list(feeder_model.graph[cim.Feeder].values())[0]
    create_distributed_feeder(feeder_model, filename)


In [ ]:
list(feeder_model.graph[cim.Feeder].values())[0]

In [ ]:
from cimtopology.distribution import create_distributed_feeder

create_distributed_feeder(feeder_model, 'ieee13_distributed.xml')

In [ ]:
feeder_model.graph[cim.Substation][UUID('6c62c905-6fc7-653d-9f1e-1340f974a587')].pprint()

In [ ]:
utils.write_xml(feeder_model, 'ieee13_distributed.xml')

In [ ]:
from cimgraph.databases import BlazegraphConnection
from cimloader.databases import BlazegraphConnection as BG
from cimloader.uploaders import BlazegraphUploader

params = ConnectionParameters(url = "http://localhost:8889/bigdata/namespace/kb/sparql", cim_profile='cimhub_2023', iec61970_301=8)
bg = BG(params)
loader = BlazegraphUploader(params)
blazegraph = BlazegraphConnection(params)
# bg.drop_all()
for filename in paths:
    loader.upload_from_file(filepath="", filename=filename)

# loader.upload_from_file(filepath='./', filename='ieee13_distributed.xml')
# loader.upload_from_file(filepath="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13", filename="IEEE13.xml")

# loader.upload_from_file(filepath="/home/ande188/Powergrid-Models/models/feeders/CIM/XML/IEEE13", filename="measurements.xml")

In [ ]:
from cimgraph.models import BusBranchModel, NodeBreakerModel
new_sub = cim.Substation()
new_sub.uuid(uri = "783e7bbe-1c15-4030-91f3-cb13e3c2ef92")
sub_model = DistributedArea(container=new_sub,
                                connection=blazegraph, distributed=False)


sub_model.get_all_edges(cim.Substation)
sub_model.get_all_edges(cim.ConnectivityNode)
sub_model.get_all_edges(cim.Terminal)
# sub_msg['AddressableEquipment'] = utils.identify_addressable(sub_model)
# sub_msg['UnaddressableEquipment'] = utils.identify_unaddressable(sub_model)
# sub_msg['Measurements'] = utils.identify_measurements(sub_model)

In [ ]:
feeder= cim.Feeder()
feeder.uuid(uri = "5814778b-9acc-4444-a823-f5c6b0861be6")
feeder_model = FeederModel(container=feeder, connection=blazegraph, distributed=True)
                
fdr_msg = json.loads(feeder.__repr__())
# try:
feeder_area_model = feeder_model.distributed_areas[0]
feeder_area_model.get_all_edges(cim.FeederArea)

In [ ]:
import cimgraph.queries.sparql as sparql
area = feeder_area_model.graph[cim.FeederArea][UUID('98aab513-615a-4ab0-b178-3c2aebd56ca4')]
print(sparql.get_all_nodes_from_area(area, blazegraph.connection_params))

In [ ]:
feeder_area_model.graph[cim.FeederArea][UUID('98aab513-615a-4ab0-b178-3c2aebd56ca4')].__dict__

In [ ]:
from cimtopology.utils import identify_measurements
identify_measurements(sub_model)


In [ ]:
print(sub_model.get_edges_query(cim.Substation))

In [ ]:
import os, json
from gridappsd import GridAPPSD
os.environ['GRIDAPPSD_USER'] = 'app_user'
os.environ['GRIDAPPSD_PASSWORD'] = '1234App'
gapps = GridAPPSD()
assert gapps.connected


In [ ]:
topic = "goss.gridappsd.request.data.cimtopology"

message = {
   "requestType": "GET_DISTRIBUTED_AREAS",
   "mRID":  "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62",
   "resultFormat": "JSON"
}

topo_message = gapps.get_response(topic, message, timeout=30)

with open('ieee13_topo_msg.json', 'w') as file:
    json.dump(topo_message, file, indent=4)

# print(json.dumps(topo_message, indent=4))


In [ ]:
message = {
   "requestType": "GET_DISTRIBUTED_AREAS",
   "mRID":  "067248c5-53c4-43e8-8720-f0c605c05b84",
   "resultFormat": "JSON"
}

topo_message = gapps.get_response(topic, message, timeout=30)

with open('vec_topo_msg.json', 'w') as file:
    json.dump(topo_message, file, indent=4)

In [ ]:

dist_area = utils.create_object(cim.DistributionArea, topo_message['DistributionArea']['@id'] )

dist_model = DistributedArea(container=dist_area, connection=blazegraph, distributed=True)

for json_ld in topo_message['DistributionArea']['Substations']:
    print('Next Substation')
    substation = dist_model.add_jsonld_to_graph(json_ld)
    SubstationArea = DistributedArea(container=substation, connection=blazegraph, distributed=True)
    SubstationArea.get_all_edges(cim.Substation)
    SubstationArea.get_all_edges(cim.Feeder)
    SubstationArea.get_all_edges(cim.ConnectivityNode)
    SubstationArea.get_all_edges(cim.Terminal)
    SubstationArea.get_all_edges(cim.Analog)
    SubstationArea.get_all_edges(cim.Discrete)

    SubstationArea.pprint(cim.Analog)
    SubstationArea.pprint(cim.Discrete)

In [ ]:

dist_area = utils.create_object(cim.DistributionArea, topo_message['DistributionArea']['@id'] )

dist_model = DistributedArea(container=dist_area, connection=blazegraph, distributed=True)

In [ ]:
SubstationArea.get_all_edges(cim.SwitchArea)
for sw_area in SubstationArea.graph.get(cim.SwitchArea, {}).values():
    print(sw_area.BoundaryTerminals)

In [ ]:
for json_ld in topo_message['DistributionArea']['Substations']:
    print('Next Substation')
    substation = dist_model.add_jsonld_to_graph(json_ld)
    SubstationArea = DistributedArea(container=substation, connection=blazegraph, distributed=True)
    SubstationArea.get_all_edges(cim.Substation)
    SubstationArea.get_all_edges(cim.Feeder)
    SubstationArea.get_all_edges(cim.ConnectivityNode)
    SubstationArea.get_all_edges(cim.Terminal)
    SubstationArea.get_all_edges(cim.Analog)
    SubstationArea.get_all_edges(cim.Discrete)

    SubstationArea.pprint(cim.Analog)
    SubstationArea.pprint(cim.Discrete)



    # name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    # print(equipment.__class__.__name__, name[0])

In [ ]:
dist_model.graph.keys()

In [ ]:
for json_ld in topo_message['DistributionArea']['FeederAreas'][0]['AddressableEquipment']:
    equipment = dist_model.add_jsonld_to_graph(json_ld)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
with open('ieee13_topo_msg.json','w') as file:
    json.dump(topo_message, file, indent = 4)

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['AddressableEquipment']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['UnaddressableEquipment']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
for mrid in topo_message['DistributionArea']['FeederAreas'][0]['Measurements']:
    equipment = blazegraph.get_object(mrid)
    name = blazegraph.get_from_triple(equipment, "IdentifiedObject.name", {})
    print(equipment.__class__.__name__, name[0])

In [ ]:
from uuid import UUID
jsonld = dict["@id":UUID,"@type":str(type)]


In [ ]:
distribution_area = blazegraph.get_object(mrid='067248c5-53c4-43e8-8720-f0c605c05b84')

In [ ]:
distribution_area_model = DistributedArea(container=distribution_area,
                                            connection=blazegraph, distributed=True)

In [ ]:
print(distribution_area_model.get_edges_query(cim.DistributionArea))

In [ ]:
distribution_area_model.get_all_edges(cim.DistributionArea)

In [ ]:
substation

In [ ]:
sub_model = DistributedArea(container=substation,
                                            connection=blazegraph, distributed=False)
sub_model.get_all_edges(cim.Substation)

In [ ]:
sub_model.graph

In [ ]:
from types import SimpleNamespace
data = SimpleNamespace(**{"@id": "John", "@type": 30})
print(data.@id)
print(data.@type)